In [ ]:
%%sql -r dataframe_1
CREATE SCHEMA IF NOT EXISTS ANYCOMPANY_LAB.ANALYTICS;

# Phase 3.1 – Création du Data Product analytique

## Contexte
À l’issue des phases 1 et 2, les données ont été intégrées puis nettoyées dans le schéma `SILVER`.  
L’objectif de cette troisième partie est de transformer ces données en **produits data analytiques réutilisables**, exploitables par :
- les équipes métier,
- les outils de visualisation,
- et, à terme, les cas d’usage de machine learning.

## Objectif
Nous allons créer un schéma `ANALYTICS` contenant plusieurs tables enrichies et agrégées permettant de piloter la performance marketing.

## Principe de modélisation
Les jeux de données disponibles couvrent plusieurs domaines métier :
- ventes,
- campagnes marketing,
- promotions,
- clients,
- avis produits,
- inventaire,
- fournisseurs.

Cependant, ils ne partagent pas tous des clés transactionnelles communes au niveau ligne-à-ligne.  
Pour garantir la cohérence métier, nous avons choisi une **approche analytique à granularité agrégée**, principalement par **région** et par **période**.

## 1. Vérification du contexte Snowflake

Avant de créer les objets analytiques, nous vérifions le rôle, la base de données, le schéma courant et le warehouse utilisé.

In [ ]:
SELECT CURRENT_ROLE(), CURRENT_DATABASE(), CURRENT_SCHEMA(), CURRENT_WAREHOUSE();

## 2. Sélection du schéma cible

Nous utilisons le schéma `ANALYTICS`, qui servira de couche de consommation analytique.

In [ ]:
USE DATABASE ANYCOMPANY_LAB;
USE SCHEMA ANALYTICS;

## 3. Création de la table `SALES_ENRICHED`

### Rôle de cette table
Cette table constitue la base analytique des ventes.  
Elle est construite à partir de `SILVER.FINANCIAL_TRANSACTIONS`, en ne conservant que les transactions de type `Sale`.

### Enrichissements apportés
Nous ajoutons :
- des variables temporelles (`year`, `month`, `quarter`, `month_date`),
- une saison,
- une catégorisation du montant,
- un indicateur de vente à forte valeur.

### Intérêt métier
Cette table permettra :
- d’analyser l’évolution des ventes dans le temps,
- de comparer les performances par région,
- d’identifier les ventes à forte contribution.

In [ ]:
CREATE OR REPLACE TABLE ANYCOMPANY_LAB.ANALYTICS.SALES_ENRICHED AS
SELECT
    TRANSACTION_ID,
    TRANSACTION_DATE,
    YEAR(TRANSACTION_DATE) AS SALES_YEAR,
    MONTH(TRANSACTION_DATE) AS SALES_MONTH,
    QUARTER(TRANSACTION_DATE) AS SALES_QUARTER,
    DATE_TRUNC('MONTH', TRANSACTION_DATE) AS SALES_MONTH_DATE,
    REGION,
    PAYMENT_METHOD,
    ENTITY,
    AMOUNT,
    CASE 
        WHEN AMOUNT < 100 THEN 'LOW'
        WHEN AMOUNT < 1000 THEN 'MEDIUM'
        ELSE 'HIGH'
    END AS SALE_AMOUNT_BAND,
    CASE
        WHEN MONTH(TRANSACTION_DATE) IN (12, 1, 2) THEN 'WINTER'
        WHEN MONTH(TRANSACTION_DATE) IN (3, 4, 5) THEN 'SPRING'
        WHEN MONTH(TRANSACTION_DATE) IN (6, 7, 8) THEN 'SUMMER'
        ELSE 'AUTUMN'
    END AS SEASON,
    CASE 
        WHEN AMOUNT >= 1000 THEN 1
        ELSE 0
    END AS HIGH_VALUE_SALE_FLAG
FROM ANYCOMPANY_LAB.SILVER.FINANCIAL_TRANSACTIONS
WHERE UPPER(TRANSACTION_TYPE) = 'SALE';

In [ ]:
SELECT * 
FROM ANYCOMPANY_LAB.ANALYTICS.SALES_ENRICHED
LIMIT 10;

## 4. Création de la table `CUSTOMER_SEGMENTS`

### Rôle de cette table
Cette table enrichit la dimension client à partir de `SILVER.CUSTOMER_DEMOGRAPHICS`.

### Enrichissements apportés
Nous calculons :
- l’âge,
- un groupe d’âge,
- un segment de revenu.

### Intérêt métier
Cette table peut être utilisée pour :
- la segmentation marketing,
- l’identification de profils à fort potentiel,
- la préparation de futures features pour des modèles ML.

In [ ]:
CREATE OR REPLACE TABLE ANYCOMPANY_LAB.ANALYTICS.CUSTOMER_SEGMENTS AS
SELECT
    CUSTOMER_ID,
    NAME,
    DATE_OF_BIRTH,
    GENDER,
    REGION,
    COUNTRY,
    CITY,
    MARITAL_STATUS,
    ANNUAL_INCOME,
    DATEDIFF('YEAR', DATE_OF_BIRTH, CURRENT_DATE()) AS AGE,
    CASE
        WHEN DATEDIFF('YEAR', DATE_OF_BIRTH, CURRENT_DATE()) < 25 THEN '18-24'
        WHEN DATEDIFF('YEAR', DATE_OF_BIRTH, CURRENT_DATE()) < 35 THEN '25-34'
        WHEN DATEDIFF('YEAR', DATE_OF_BIRTH, CURRENT_DATE()) < 50 THEN '35-49'
        WHEN DATEDIFF('YEAR', DATE_OF_BIRTH, CURRENT_DATE()) < 65 THEN '50-64'
        ELSE '65+'
    END AS AGE_GROUP,
    CASE
        WHEN ANNUAL_INCOME < 30000 THEN 'LOW INCOME'
        WHEN ANNUAL_INCOME < 70000 THEN 'MIDDLE INCOME'
        WHEN ANNUAL_INCOME < 120000 THEN 'UPPER MIDDLE INCOME'
        ELSE 'HIGH INCOME'
    END AS INCOME_SEGMENT
FROM ANYCOMPANY_LAB.SILVER.CUSTOMER_DEMOGRAPHICS;

In [ ]:
SELECT * 
FROM ANYCOMPANY_LAB.ANALYTICS.CUSTOMER_SEGMENTS
LIMIT 10;

## 5. Création de la table `CAMPAIGNS_ENRICHED`

### Rôle de cette table
Cette table enrichit les campagnes marketing en ajoutant des indicateurs de performance simples.

### Enrichissements apportés
Nous calculons :
- la durée de la campagne,
- le coût par portée (`cost_per_reach`),
- les conversions estimées.

### Intérêt métier
Cette table permet :
- d’évaluer l’efficience des campagnes,
- de comparer les régions et les catégories,
- de structurer un pilotage marketing plus data-driven.

In [ ]:
CREATE OR REPLACE TABLE ANYCOMPANY_LAB.ANALYTICS.CAMPAIGNS_ENRICHED AS
SELECT
    CAMPAIGN_ID,
    CAMPAIGN_NAME,
    CAMPAIGN_TYPE,
    PRODUCT_CATEGORY,
    TARGET_AUDIENCE,
    START_DATE,
    END_DATE,
    DATE_TRUNC('MONTH', START_DATE) AS CAMPAIGN_MONTH_DATE,
    REGION,
    BUDGET,
    REACH,
    CONVERSION_RATE,
    DATEDIFF('DAY', START_DATE, END_DATE) + 1 AS CAMPAIGN_DURATION_DAYS,
    CASE 
        WHEN REACH > 0 THEN BUDGET / REACH
        ELSE NULL
    END AS COST_PER_REACH,
    ROUND(REACH * CONVERSION_RATE, 0) AS ESTIMATED_CONVERSIONS
FROM ANYCOMPANY_LAB.SILVER.MARKETING_CAMPAIGNS;

In [ ]:
SELECT * 
FROM ANYCOMPANY_LAB.ANALYTICS.CAMPAIGNS_ENRICHED
LIMIT 10;

## 6. Création de la table `PROMOTIONS_ENRICHED`

### Rôle de cette table
Cette table enrichit les promotions pour faciliter leur analyse.

### Enrichissements apportés
Nous calculons :
- le mois de démarrage de la promotion,
- la durée de la promotion,
- une intensité de remise (`low`, `medium`, `high`).

### Intérêt métier
Elle permettra d’analyser :
- la pression promotionnelle,
- la fréquence des promotions,
- le niveau moyen de remise selon les régions et les périodes.

In [ ]:
CREATE OR REPLACE TABLE ANYCOMPANY_LAB.ANALYTICS.PROMOTIONS_ENRICHED AS
SELECT
    PROMOTION_ID,
    PRODUCT_CATEGORY,
    PROMOTION_TYPE,
    DISCOUNT_PERCENTAGE,
    START_DATE,
    END_DATE,
    DATE_TRUNC('MONTH', START_DATE) AS PROMO_MONTH_DATE,
    REGION,
    DATEDIFF('DAY', START_DATE, END_DATE) + 1 AS PROMOTION_DURATION_DAYS,
    CASE
        WHEN DISCOUNT_PERCENTAGE < 0.10 THEN 'LOW'
        WHEN DISCOUNT_PERCENTAGE < 0.20 THEN 'MEDIUM'
        ELSE 'HIGH'
    END AS PROMOTION_INTENSITY
FROM ANYCOMPANY_LAB.SILVER.PROMOTIONS_CLEAN;

In [ ]:
SELECT * 
FROM ANYCOMPANY_LAB.ANALYTICS.PROMOTIONS_ENRICHED
LIMIT 10;

## 7. Création de la table `REVIEWS_AGGREGATED`

### Rôle de cette table
Les avis produits ne peuvent pas être reliés directement aux ventes au niveau transactionnel.  
Nous les agrégeons donc à une granularité analytique exploitable.

### Granularité choisie
- mois,
- catégorie produit.

### Intérêt métier
Cette table permet de suivre :
- le volume d’avis,
- la note moyenne par catégorie,
- l’évolution de la perception produit dans le temps.

In [ ]:
CREATE OR REPLACE TABLE ANYCOMPANY_LAB.ANALYTICS.REVIEWS_AGGREGATED AS
SELECT
    DATE_TRUNC('MONTH', REVIEW_DATE) AS REVIEW_MONTH_DATE,
    PRODUCT_CATEGORY,
    COUNT(*) AS NB_REVIEWS,
    AVG(RATING) AS AVG_RATING
FROM ANYCOMPANY_LAB.SILVER.PRODUCT_REVIEWS
GROUP BY 1, 2;

In [ ]:
SELECT * 
FROM ANYCOMPANY_LAB.ANALYTICS.REVIEWS_AGGREGATED
LIMIT 10;

## 8. Création de la table `INVENTORY_AGGREGATED`

### Rôle de cette table
Cette table synthétise l’état du stock par région et par catégorie produit.

### Indicateurs calculés
Nous calculons :
- le nombre de produits en stock,
- le stock moyen,
- le point de réapprovisionnement moyen,
- le délai moyen,
- le nombre d’alertes de stock.

### Intérêt métier
Cette table permet de faire le lien entre performance marketing et contraintes opérationnelles.

In [ ]:
CREATE OR REPLACE TABLE ANYCOMPANY_LAB.ANALYTICS.INVENTORY_AGGREGATED AS
SELECT
    REGION,
    PRODUCT_CATEGORY,
    COUNT(*) AS NB_PRODUCTS_IN_STOCK,
    AVG(CURRENT_STOCK) AS AVG_CURRENT_STOCK,
    AVG(REORDER_POINT) AS AVG_REORDER_POINT,
    AVG(LEAD_TIME) AS AVG_LEAD_TIME,
    SUM(CASE WHEN CURRENT_STOCK <= REORDER_POINT THEN 1 ELSE 0 END) AS STOCK_ALERT_COUNT
FROM ANYCOMPANY_LAB.SILVER.INVENTORY
GROUP BY 1, 2;

In [ ]:
SELECT * 
FROM ANYCOMPANY_LAB.ANALYTICS.INVENTORY_AGGREGATED
LIMIT 10;

## 9. Création de la table `SUPPLIER_INVENTORY_SUMMARY`

### Rôle de cette table
Cette table combine les informations fournisseurs et les indicateurs d’inventaire à une granularité commune :
- région,
- catégorie produit.

### Intérêt métier
Elle permet d’analyser si certaines difficultés logistiques ou certains niveaux de stock peuvent être expliqués par :
- le délai fournisseur,
- la fiabilité fournisseur,
- la qualité fournisseur.

In [ ]:
CREATE OR REPLACE TABLE ANYCOMPANY_LAB.ANALYTICS.SUPPLIER_INVENTORY_SUMMARY AS
SELECT
    s.REGION,
    s.PRODUCT_CATEGORY,
    COUNT(DISTINCT s.SUPPLIER_ID) AS NB_SUPPLIERS,
    AVG(s.LEAD_TIME) AS AVG_SUPPLIER_LEAD_TIME,
    AVG(s.RELIABILITY_SCORE) AS AVG_RELIABILITY_SCORE,
    COUNT_IF(s.QUALITY_RATING = 'A') AS NB_QUALITY_A_SUPPLIERS,
    i.NB_PRODUCTS_IN_STOCK,
    i.AVG_CURRENT_STOCK,
    i.AVG_LEAD_TIME AS AVG_INVENTORY_LEAD_TIME,
    i.STOCK_ALERT_COUNT
FROM ANYCOMPANY_LAB.SILVER.SUPPLIER_INFORMATION s
LEFT JOIN ANYCOMPANY_LAB.ANALYTICS.INVENTORY_AGGREGATED i
    ON s.REGION = i.REGION
   AND s.PRODUCT_CATEGORY = i.PRODUCT_CATEGORY
GROUP BY
    s.REGION,
    s.PRODUCT_CATEGORY,
    i.NB_PRODUCTS_IN_STOCK,
    i.AVG_CURRENT_STOCK,
    i.AVG_LEAD_TIME,
    i.STOCK_ALERT_COUNT;

In [ ]:
SELECT * 
FROM ANYCOMPANY_LAB.ANALYTICS.SUPPLIER_INVENTORY_SUMMARY
LIMIT 10;

## 10. Création de la table centrale `MARKETING_PERFORMANCE_MART`

### Rôle de cette table
Il s’agit du **data product central** de cette partie.

### Granularité retenue
Nous avons choisi la granularité :
- **mois × région**

### Pourquoi cette granularité ?
Les différentes tables disponibles ne partagent pas toutes des clés transactionnelles communes.  
En revanche, elles partagent des dimensions fiables :
- la région,
- la période.

Cette granularité permet donc de :
- consolider les ventes,
- relier les campagnes marketing,
- relier les promotions,
- fournir une table stable et réutilisable pour le pilotage métier.

### Indicateurs inclus
- ventes,
- campagnes,
- budget marketing,
- portée,
- conversions estimées,
- promotions,
- taux de remise moyen.

In [ ]:
CREATE OR REPLACE TABLE ANYCOMPANY_LAB.ANALYTICS.MARKETING_PERFORMANCE_MART AS
WITH sales_monthly AS (
    SELECT
        SALES_MONTH_DATE AS MONTH_DATE,
        REGION,
        COUNT(*) AS NB_SALES,
        SUM(AMOUNT) AS TOTAL_SALES_AMOUNT,
        AVG(AMOUNT) AS AVG_SALES_AMOUNT,
        SUM(HIGH_VALUE_SALE_FLAG) AS NB_HIGH_VALUE_SALES
    FROM ANYCOMPANY_LAB.ANALYTICS.SALES_ENRICHED
    GROUP BY 1, 2
),
campaign_monthly AS (
    SELECT
        CAMPAIGN_MONTH_DATE AS MONTH_DATE,
        REGION,
        COUNT(*) AS NB_CAMPAIGNS,
        SUM(BUDGET) AS TOTAL_CAMPAIGN_BUDGET,
        SUM(REACH) AS TOTAL_CAMPAIGN_REACH,
        AVG(CONVERSION_RATE) AS AVG_CONVERSION_RATE,
        SUM(ESTIMATED_CONVERSIONS) AS TOTAL_ESTIMATED_CONVERSIONS
    FROM ANYCOMPANY_LAB.ANALYTICS.CAMPAIGNS_ENRICHED
    GROUP BY 1, 2
),
promo_monthly AS (
    SELECT
        PROMO_MONTH_DATE AS MONTH_DATE,
        REGION,
        COUNT(*) AS NB_PROMOTIONS,
        AVG(DISCOUNT_PERCENTAGE) AS AVG_DISCOUNT_PERCENTAGE
    FROM ANYCOMPANY_LAB.ANALYTICS.PROMOTIONS_ENRICHED
    GROUP BY 1, 2
)
SELECT
    COALESCE(s.MONTH_DATE, c.MONTH_DATE, p.MONTH_DATE) AS MONTH_DATE,
    COALESCE(s.REGION, c.REGION, p.REGION) AS REGION,
    COALESCE(s.NB_SALES, 0) AS NB_SALES,
    COALESCE(s.TOTAL_SALES_AMOUNT, 0) AS TOTAL_SALES_AMOUNT,
    COALESCE(s.AVG_SALES_AMOUNT, 0) AS AVG_SALES_AMOUNT,
    COALESCE(s.NB_HIGH_VALUE_SALES, 0) AS NB_HIGH_VALUE_SALES,
    COALESCE(c.NB_CAMPAIGNS, 0) AS NB_CAMPAIGNS,
    COALESCE(c.TOTAL_CAMPAIGN_BUDGET, 0) AS TOTAL_CAMPAIGN_BUDGET,
    COALESCE(c.TOTAL_CAMPAIGN_REACH, 0) AS TOTAL_CAMPAIGN_REACH,
    COALESCE(c.AVG_CONVERSION_RATE, 0) AS AVG_CONVERSION_RATE,
    COALESCE(c.TOTAL_ESTIMATED_CONVERSIONS, 0) AS TOTAL_ESTIMATED_CONVERSIONS,
    COALESCE(p.NB_PROMOTIONS, 0) AS NB_PROMOTIONS,
    COALESCE(p.AVG_DISCOUNT_PERCENTAGE, 0) AS AVG_DISCOUNT_PERCENTAGE
FROM sales_monthly s
FULL OUTER JOIN campaign_monthly c
    ON s.MONTH_DATE = c.MONTH_DATE
   AND s.REGION = c.REGION
FULL OUTER JOIN promo_monthly p
    ON COALESCE(s.MONTH_DATE, c.MONTH_DATE) = p.MONTH_DATE
   AND COALESCE(s.REGION, c.REGION) = p.REGION;

In [ ]:
SELECT * 
FROM ANYCOMPANY_LAB.ANALYTICS.MARKETING_PERFORMANCE_MART
ORDER BY MONTH_DATE, REGION
LIMIT 20;

## 11. Création d’une vue métier

Afin de faciliter la consommation par des outils analytiques ou des dashboards, nous créons une vue simplifiée à partir du mart principal.

In [ ]:
CREATE OR REPLACE VIEW ANYCOMPANY_LAB.ANALYTICS.V_MARKETING_PERFORMANCE AS
SELECT
    MONTH_DATE,
    REGION,
    NB_SALES,
    TOTAL_SALES_AMOUNT,
    AVG_SALES_AMOUNT,
    NB_HIGH_VALUE_SALES,
    NB_CAMPAIGNS,
    TOTAL_CAMPAIGN_BUDGET,
    TOTAL_CAMPAIGN_REACH,
    AVG_CONVERSION_RATE,
    TOTAL_ESTIMATED_CONVERSIONS,
    NB_PROMOTIONS,
    AVG_DISCOUNT_PERCENTAGE
FROM ANYCOMPANY_LAB.ANALYTICS.MARKETING_PERFORMANCE_MART;

In [ ]:
SELECT * 
FROM ANYCOMPANY_LAB.ANALYTICS.V_MARKETING_PERFORMANCE
LIMIT 20;

## 12. Vérification finale

Nous vérifions maintenant que les objets du schéma `ANALYTICS` ont bien été créés.

In [ ]:
SHOW TABLES IN SCHEMA ANYCOMPANY_LAB.ANALYTICS;

In [ ]:
SHOW VIEWS IN SCHEMA ANYCOMPANY_LAB.ANALYTICS;

## Conclusion

Cette partie 3.1 a permis de transformer les analyses exploratoires des phases précédentes en un **produit data structuré, stable et réutilisable**.

### Résultats obtenus
Nous avons créé :
- des tables enrichies pour les ventes, les clients, les campagnes et les promotions,
- des tables agrégées pour les avis, l’inventaire et les fournisseurs,
- un mart central `MARKETING_PERFORMANCE_MART`,
- une vue métier de consommation.

### Valeur ajoutée
Ce produit data peut désormais être utilisé pour :
- le reporting marketing,
- l’analyse de performance par région et par période,
- la visualisation dans Streamlit,
- et, à terme, le feature engineering ou des cas d’usage machine learning.

In [ ]:
SELECT
    REGION,
    SUM(TOTAL_SALES_AMOUNT) AS SALES,
    SUM(TOTAL_CAMPAIGN_BUDGET) AS BUDGET,
    AVG(AVG_CONVERSION_RATE) AS AVG_CONV_RATE
FROM ANYCOMPANY_LAB.ANALYTICS.MARKETING_PERFORMANCE_MART
GROUP BY REGION
ORDER BY SALES DESC;

### Lecture rapide
Cette requête permet d’obtenir une première lecture consolidée de la performance marketing par région.